In [ ]:
import copy
import inspect
import json
import math
import pickle
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "training_data"

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())

In [ ]:
# Data and target settings
DATASET_NAME = "mean_curvature_smooth"
TARGET_INDICES = [0]
TARGET_INDEX_FOR_PLOTS = 0
USE_GLOBAL_FEATURES = True

# Filtering and preprocessing settings
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
COMPLEXITY_MIN = 2.0
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Split and reference-organoid settings
VAL_FRAC = 0.2
SPLIT_SEED = None
FORCED_VAL_KEYS = {
    ("20251201", "day4p5_B03_144"),
}
REFERENCE_VIEW = dict(azim=-135, elev=55)

# Feature-subset experiment settings
FEATURE_VARIANTS = {
    "all_markers": None,
    "LGR5_only": ["LGR5"],
    "LGR5_Lysozyme_Serotonin_Chroma": ["LGR5", "Lysozyme", "Serotonin", "Chroma"],
}
FEATURE_VARIANT_LABELS = {
    "all_markers": "All markers",
    "LGR5_only": "LGR5 only",
    "LGR5_Lysozyme_Serotonin_Chroma": "LGR5 + Lysozyme + Serotonin + Chroma",
}
DEPTHS = [0, 1, 2, 3, 4]

# Model settings
HIDDEN_DIM = 4 * 64
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True

# Training settings
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 2000
PATIENCE = 30
NUM_WORKERS = 4
PREDICT_BATCH_SIZE = 128
EDGE_LOSS_WEIGHT = 0.20
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}

# Experiment output settings
EXPERIMENT_GROUP = "marker_subset_depth_comparison"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"run_{RUN_TIMESTAMP}"
SAVE_DIR = PROJECT_ROOT / "results_experiments" / EXPERIMENT_GROUP / RUN_NAME
FIGURES_DIR = SAVE_DIR / "figures"

# Marker Subset Depth Comparison

This experiment trains depth-scanned GIN models on the same train/validation organoids while varying only the available fate-marker columns.

In [ ]:
def _safe_filename(name):
    text = str(name).strip().replace("/", "_")
    chars = [ch if (ch.isalnum() or ch in "._-") else "_" for ch in text]
    cleaned = "".join(chars).strip("._-")
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned or "figure"


def ensure_figure_dir():
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    return FIGURES_DIR


def save_mpl_figure(fig, name, *, close=False):
    path = ensure_figure_dir() / f"{_safe_filename(name)}.pdf"
    fig.savefig(path, bbox_inches="tight", transparent=True)
    print(f"Saved figure -> {path}")
    if close:
        plt.close(fig)
    return path


def save_plotly_figure(fig, name, *, scale=2):
    path = ensure_figure_dir() / f"{_safe_filename(name)}.png"
    try:
        fig.write_image(str(path), scale=scale)
        print(f"Saved Plotly figure -> {path}")
    except Exception as exc:
        print(f"Could not save Plotly figure as PNG at {path}: {exc}")
    return path


def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, np.dtype):
        return str(obj)
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_joint_experiment(save_dir, config, results_by_experiment, notes=None):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    with open(save_dir / "config.json", "w") as f:
        json.dump(_jsonable(config), f, indent=2)

    with open(save_dir / "results.pkl", "wb") as f:
        pickle.dump(results_by_experiment, f)

    meta = {
        "timestamp": datetime.now().isoformat(),
        "notes": notes,
        "experiments": list(results_by_experiment.keys()),
    }
    with open(save_dir / "meta.json", "w") as f:
        json.dump(_jsonable(meta), f, indent=2)

    print(f"Saved joint experiment -> {save_dir}")

## Load Data

In [ ]:
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    attach_metadata_to_graphs,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
    print_graph_and_metadata_fields,
)

data_dir = DATA_ROOT / DATASET_NAME
graphs = load_graph_dataset_from_dir(str(data_dir))
print(f"Loaded {len(graphs)} organoids.")
if graphs:
    print("Raw y shape:", tuple(graphs[0].y.shape))

meta = load_aux_metadata_for_dir(str(data_dir))
attached = attach_metadata_to_graphs(graphs, meta, exclude_keys=None)
print(f"Attached metadata to {attached}/{len(graphs)} graphs.")

graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
if graphs:
    print("Selected y shape:", tuple(graphs[0].y.shape))

marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    n_markers = int(graphs[0].x.size(1)) if graphs else 0
    marker_names = [f"marker_{i}" for i in range(n_markers)]
print(f"Loaded {len(marker_names)} markers:", marker_names)

print_graph_and_metadata_fields(graphs)

## Filter And Preprocess

In [ ]:
from src.data.metadata import fill_missing_metadata_for_group
from src.data.filters import (
    filter_graphs_by_marker_diversity,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors

graphs = fill_missing_metadata_for_group(
    graphs,
    field="complexity",
    fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
    dataset=MISSING_COMPLEXITY_GROUP["dataset"],
    timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
)

graphs, g_spherical = filter_graphs_by_sphericity(
    graphs,
    max_sphericity=SPHERICITY_MAX,
    print_summary=True,
    return_rejected=True,
)

g_spherical = filter_graphs_by_marker_diversity(
    g_spherical,
    min_score=SPHERICAL_MARKER_DIVERSITY_MIN,
    print_summary=True,
)

graphs = filter_graphs_by_numeric_metadata(
    graphs,
    key="complexity",
    min_value=COMPLEXITY_MIN,
    allow_missing=False,
    inplace=False,
    print_summary=True,
)

graphs = graphs + g_spherical
print(f"After filtering and spherical rescue: {len(graphs)} organoids.")

if INTERPOLATE_TARGET_OUTLIERS:
    graphs, outlier_info = interpolate_target_outliers_from_neighbors(
        graphs,
        target_indices=None,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
else:
    outlier_info = None

In [ ]:
from src.data.metadata import add_log_metadata_features, promote_metadata_to_graph_tensors

field_specs = [
    {
        "meta_keys": [
            "log_surface_area",
            "log_volume",
            "log_volume_over_area",
            "log_num_cells",
        ],
        "attr_name": "global_feat",
        "kind": "graph_vector",
        "dtype": torch.float32,
    },
]

if USE_GLOBAL_FEATURES:
    graphs = add_log_metadata_features(graphs, inplace=False)
    graphs = promote_metadata_to_graph_tensors(graphs, field_specs, inplace=False)
    print("Promoted metadata fields to graph tensor attributes.")
else:
    print("Global features disabled; no global_feat attribute was attached.")

## Shared Split And Feature Variants

In [ ]:
from src.data.splits import graph_metadata_key, train_val_split_graphs
from src.data.metadata import infer_global_dim, snapshot_graph_metadata, strip_graph_metadata
from src.data.target_transforms import AsinhStandardizeTransform, standardize_graph_global_features

g_train, g_val, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    force_val_keys=FORCED_VAL_KEYS,
    key_fn=graph_metadata_key,
)
print(f"Split -> train: {len(g_train)} | val: {len(g_val)}")

train_meta_lookup = snapshot_graph_metadata(g_train)
val_meta_lookup = snapshot_graph_metadata(g_val)

g_train = strip_graph_metadata(g_train, inplace=False)
g_val = strip_graph_metadata(g_val, inplace=False)

target_transform = AsinhStandardizeTransform(robust=True).fit(g_train)
target_transform.transform_graphs(g_train)
target_transform.transform_graphs(g_val)

center_global, scale_global = None, None
if USE_GLOBAL_FEATURES:
    center_global, scale_global = standardize_graph_global_features(
        g_train,
        g_val,
        attr_name="global_feat",
        robust=False,
    )

global_dim = infer_global_dim(g_train)
print("global_dim =", global_dim)
print("target_transform =", target_transform.name)

In [ ]:
from src.data.preprocessing import remove_marker_features


def _resolve_marker_name(name, marker_names):
    marker_names = list(marker_names)
    if name in marker_names:
        return marker_names.index(name)

    lower_lookup = {str(m).lower(): i for i, m in enumerate(marker_names)}
    key = str(name).lower()
    if key not in lower_lookup:
        raise ValueError(f"Marker {name!r} not found in marker_names: {marker_names}")
    return lower_lookup[key]


def build_feature_variant(graphs_in, keep_markers, marker_names):
    marker_names = list(marker_names)
    if keep_markers is None:
        return [copy.copy(g) for g in graphs_in], marker_names

    keep_indices = {_resolve_marker_name(name, marker_names) for name in keep_markers}
    remove_names = [name for i, name in enumerate(marker_names) if i not in keep_indices]
    graphs_out, marker_names_out = remove_marker_features(
        graphs_in,
        remove_names,
        marker_names=marker_names,
        inplace=False,
        return_marker_names=True,
    )
    return graphs_out, marker_names_out


feature_data = {}
for variant_name, keep_markers in FEATURE_VARIANTS.items():
    train_variant, marker_names_variant = build_feature_variant(g_train, keep_markers, marker_names)
    val_variant, marker_names_val_variant = build_feature_variant(g_val, keep_markers, marker_names)

    if marker_names_variant != marker_names_val_variant:
        raise RuntimeError(f"Train/val marker mismatch for {variant_name}")

    feature_data[variant_name] = {
        "g_train": train_variant,
        "g_val": val_variant,
        "marker_names": marker_names_variant,
        "keep_markers": None if keep_markers is None else list(keep_markers),
    }

    print(
        f"{variant_name:<36} "
        f"features={train_variant[0].x.size(1):>2} | "
        f"markers={marker_names_variant}"
    )

## Train Depth-Scanned Models

In [ ]:
from src.models.gnn import GINCurvature
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term

aux_losses = [
    WeightedLossTerm(
        name="edge",
        fn=edge_loss_term,
        weight=EDGE_LOSS_WEIGHT,
        params=EDGE_LOSS_PARAMS,
    ),
]

cfg = TrainConfig(
    lr=LR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    aux_losses=aux_losses,
)

device = cfg.device if "cfg" in globals() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

In [ ]:
trained_models = {}
training_logs = {}

for variant_name, pack in feature_data.items():
    for depth in DEPTHS:
        print()
        print("=" * 80)
        print(f"Training GINCurvature | variant={variant_name} | depth={depth}")

        model = GINCurvature(
            n_markers=int(pack["g_train"][0].x.size(1)),
            global_dim=infer_global_dim(pack["g_train"]),
            hidden_dim=HIDDEN_DIM,
            num_layers=int(depth),
            dropout=DROPOUT,
            residual=RESIDUAL,
            norm=NORM,
        )

        model, metrics, history = train(model, pack["g_train"], pack["g_val"], cfg)
        trained_models[(variant_name, int(depth))] = model
        training_logs[(variant_name, int(depth))] = {
            "variant": variant_name,
            "depth": int(depth),
            "metrics": metrics,
            "history": history,
        }

print("Finished training", len(trained_models), "models.")

## Validation Error

In [ ]:
from src.inference.predict import predict_targets


def select_target_column(values, target_index=0):
    arr = np.asarray(values)
    if arr.ndim == 1:
        return arr.reshape(-1)
    if arr.ndim == 2 and arr.shape[1] == 1:
        return arr[:, 0]
    if arr.ndim == 2:
        return arr[:, int(target_index)]
    raise ValueError(f"Expected 1-D or 2-D target array, got shape {arr.shape}")


predictions_by_variant_depth = {}
mse_rows = []
truth_reference = None

for variant_name, pack in feature_data.items():
    for depth in DEPTHS:
        model = trained_models[(variant_name, int(depth))]
        y_true, y_pred, X_pred = predict_targets(
            pack["g_val"],
            model,
            device=device,
            batch_size=PREDICT_BATCH_SIZE,
            target_transform=target_transform,
        )

        y_true_plot = select_target_column(y_true, TARGET_INDEX_FOR_PLOTS)
        y_pred_plot = select_target_column(y_pred, TARGET_INDEX_FOR_PLOTS)
        sq_err = (y_pred_plot - y_true_plot) ** 2

        if truth_reference is None:
            truth_reference = y_true_plot.copy()
        elif not np.allclose(truth_reference, y_true_plot, equal_nan=True):
            raise RuntimeError("Validation truth differs between feature variants; split/base graphs are not aligned.")

        predictions_by_variant_depth[(variant_name, int(depth))] = {
            "y_true": y_true_plot,
            "y_pred": y_pred_plot,
            "x": X_pred,
        }
        mse_rows.append(
            {
                "variant": variant_name,
                "variant_label": FEATURE_VARIANT_LABELS.get(variant_name, variant_name),
                "depth": int(depth),
                "mse": float(np.mean(sq_err)),
                "sem": float(np.std(sq_err, ddof=1) / np.sqrt(len(sq_err))) if len(sq_err) > 1 else 0.0,
                "n_nodes": int(len(sq_err)),
            }
        )

mse_df = pd.DataFrame(mse_rows).sort_values(["variant", "depth"]).reset_index(drop=True)
display(mse_df)

In [ ]:
baseline_sq_err = (truth_reference - np.mean(truth_reference)) ** 2
baseline_mse = float(np.mean(baseline_sq_err))
baseline_sem = float(np.std(baseline_sq_err, ddof=1) / np.sqrt(len(baseline_sq_err)))

fig, ax = plt.subplots(figsize=(7.5, 4.8))
colors = plt.get_cmap("tab10").colors

for color_i, variant_name in enumerate(FEATURE_VARIANTS.keys()):
    sub = mse_df[mse_df["variant"] == variant_name].sort_values("depth")
    label = FEATURE_VARIANT_LABELS.get(variant_name, variant_name)
    color = colors[color_i % len(colors)]
    ax.plot(sub["depth"], sub["mse"], "-o", linewidth=2, markersize=5, label=label, color=color)
    ax.fill_between(
        sub["depth"].to_numpy(),
        (sub["mse"] - sub["sem"]).to_numpy(),
        (sub["mse"] + sub["sem"]).to_numpy(),
        alpha=0.16,
        color=color,
    )

ax.axhline(baseline_mse, linestyle="--", linewidth=1.8, color="gray", label="validation mean baseline")
ax.fill_between(
    [min(DEPTHS), max(DEPTHS)],
    [baseline_mse - baseline_sem, baseline_mse - baseline_sem],
    [baseline_mse + baseline_sem, baseline_mse + baseline_sem],
    color="gray",
    alpha=0.12,
)

ax.set_xlabel("Model depth / number of GNN layers")
ax.set_ylabel("Validation MSE")
ax.set_title("Validation MSE by marker subset and model depth")
ax.set_xticks(DEPTHS)
ax.legend(frameon=False)

plt.tight_layout()
save_mpl_figure(fig, "validation_mse_by_marker_subset_and_depth")
plt.show()

## Reference Organoid Predictions

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from organograph.plotting.meshes import plot_organoid_mesh

from src.data.splits import select_graphs_by_keys
from src.plotting.mesh_plots import get_graph_slice_bounds, project_node_quantities_to_mesh

selected_graphs = select_graphs_by_keys(
    g_val,
    FORCED_VAL_KEYS,
    key_fn=graph_metadata_key,
    meta_lookup=val_meta_lookup,
)

if selected_graphs:
    g_ref = selected_graphs[0]
else:
    print("No forced reference organoid found in validation; using the first validation graph.")
    g_ref = g_val[0]

selected_organoid_str = getattr(g_ref, "organoid_str")
graph_index = [
    i for i, gg in enumerate(g_val)
    if getattr(gg, "organoid_str", None) == selected_organoid_str
][0]
start, end = get_graph_slice_bounds(g_val, graph_index)
selected_key = graph_metadata_key(g_ref, meta_lookup=val_meta_lookup)
print("Reference organoid:", selected_key, selected_organoid_str, "node slice", (start, end))

truth_ref = truth_reference[start:end]
all_ref_values = [truth_ref]
for variant_name in FEATURE_VARIANTS:
    for depth in DEPTHS:
        all_ref_values.append(predictions_by_variant_depth[(variant_name, int(depth))]["y_pred"][start:end])
all_ref_values = np.concatenate([np.asarray(v).reshape(-1) for v in all_ref_values])
finite_ref_values = all_ref_values[np.isfinite(all_ref_values)]
abs_max = float(np.max(np.abs(finite_ref_values))) if finite_ref_values.size else 1.0
vmin, vmax = -abs_max, abs_max

In [ ]:
variant_order = list(FEATURE_VARIANTS.keys())
n_rows = len(variant_order)
n_cols = len(DEPTHS) + 1

subplot_titles = []
for variant_name in variant_order:
    label = FEATURE_VARIANT_LABELS.get(variant_name, variant_name)
    subplot_titles.append(f"{label}<br>true")
    for depth in DEPTHS:
        subplot_titles.append(f"{label}<br>depth={depth}")

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    specs=[[{"type": "scene"} for _ in range(n_cols)] for _ in range(n_rows)],
    subplot_titles=subplot_titles,
    horizontal_spacing=0.01,
    vertical_spacing=0.03,
)

for row, variant_name in enumerate(variant_order, start=1):
    for col, depth_or_true in enumerate(["true"] + list(DEPTHS), start=1):
        if depth_or_true == "true":
            node_pred = truth_ref
            node_true = truth_ref
            values = "mesh_true"
        else:
            node_pred = predictions_by_variant_depth[(variant_name, int(depth_or_true))]["y_pred"][start:end]
            node_true = truth_ref
            values = "mesh_pred"

        proj = project_node_quantities_to_mesh(
            g_ref,
            node_pred=node_pred,
            node_true=node_true,
            target_index=None,
            meta_lookup=val_meta_lookup,
        )
        subfig = plot_organoid_mesh(
            proj["mesh"],
            vertex_values=proj[values],
            backend="plotly",
            colorscale="RdBu_r",
            center_at_zero=True,
            vmin=vmin,
            vmax=vmax,
            show_colorbar=(row == 1 and col == n_cols),
            fig_size=(450, 420),
            view=REFERENCE_VIEW,
        )

        for tr in subfig.data:
            fig.add_trace(tr, row=row, col=col)

        scene_name = "scene" if (row == 1 and col == 1) else f"scene{(row - 1) * n_cols + col}"
        if subfig.layout.scene is not None:
            fig.layout[scene_name].update(subfig.layout.scene.to_plotly_json())

fig.update_layout(
    width=360 * n_cols,
    height=330 * n_rows,
    title=f"Reference organoid predictions by marker subset and depth | {selected_key}",
    margin=dict(l=10, r=10, t=90, b=10),
)

save_plotly_figure(fig, "reference_organoid_predictions_by_marker_subset_and_depth")
fig.show()

## Save Results

In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)
mse_csv_path = SAVE_DIR / "mse_by_marker_subset_and_depth.csv"
mse_df.to_csv(mse_csv_path, index=False)
print(f"Saved MSE table -> {mse_csv_path}")

config = {
    "dataset_name": DATASET_NAME,
    "target_indices": TARGET_INDICES,
    "target_index_for_plots": TARGET_INDEX_FOR_PLOTS,
    "use_global_features": USE_GLOBAL_FEATURES,
    "filtering": {
        "missing_complexity_group": MISSING_COMPLEXITY_GROUP,
        "sphericity_max": SPHERICITY_MAX,
        "spherical_marker_diversity_min": SPHERICAL_MARKER_DIVERSITY_MIN,
        "complexity_min": COMPLEXITY_MIN,
        "interpolate_target_outliers": INTERPOLATE_TARGET_OUTLIERS,
        "outlier_clip_quantiles": OUTLIER_CLIP_QUANTILES,
        "outlier_info": outlier_info,
    },
    "split": split_info,
    "feature_variants": FEATURE_VARIANTS,
    "feature_variant_labels": FEATURE_VARIANT_LABELS,
    "depths": DEPTHS,
    "model": {
        "model_class": "GINCurvature",
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "norm": NORM,
        "residual": RESIDUAL,
        "global_dim": global_dim,
    },
    "training": {
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "num_workers": NUM_WORKERS,
        "edge_loss_weight": EDGE_LOSS_WEIGHT,
        "edge_loss_params": EDGE_LOSS_PARAMS,
    },
    "marker_names": marker_names,
    "variant_marker_names": {name: pack["marker_names"] for name, pack in feature_data.items()},
    "global_feature_center": center_global,
    "global_feature_scale": scale_global,
    "save_dir": SAVE_DIR,
    "figures_dir": FIGURES_DIR,
}

results_by_experiment = {
    "mse_table": mse_df,
    "baseline": {
        "mse": baseline_mse,
        "sem": baseline_sem,
        "n_nodes": int(len(truth_reference)),
    },
    "predictions_by_variant_depth": predictions_by_variant_depth,
    "training_logs": training_logs,
    "reference_organoid": {
        "key": selected_key,
        "organoid_str": selected_organoid_str,
        "graph_index": int(graph_index),
        "node_slice": (int(start), int(end)),
    },
}

save_joint_experiment(
    SAVE_DIR,
    config=config,
    results_by_experiment=results_by_experiment,
    notes="Depth scan comparing all markers, LGR5 only, and LGR5/Lysozyme/Serotonin/Chroma marker subsets.",
)